In [2]:
# ══════════════════════════════════════════════════════════════
# SETUP CELL — GitHub → Q-Learning (flat zip, files directly inside)
# ══════════════════════════════════════════════════════════════
import subprocess, shutil, os, zipfile, sys

# Step 1 — Clone repo
if os.path.exists('/content/repo'):
    shutil.rmtree('/content/repo')

print('Cloning from GitHub...')
subprocess.run([
    'git', 'clone',
    'https://github.com/maham-bhatti/Traffic-Control-System.git',
    '/content/repo'
], check=True)
print('✓ Repo cloned')

# Step 2 — Extract flat zip directly to /content
zip_path = '/content/repo/Q-learning.zip'
print('\nExtracting Q-Learning.zip...')
with zipfile.ZipFile(zip_path, 'r') as z:
    print('── Files inside zip ──')
    for name in sorted(z.namelist()):
        print(f'  {name}')
    z.extractall('/content/')
print('✓ Extracted directly to /content')

# Step 3 — Python path
sys.path.insert(0, '/content')

# Step 4 — Verify
REQUIRED = [
    'gcn_encoder.py', 'ma_qlearning.py', 'traci_env_qlearning.py',
    'manhattan.net.xml', 'run.sumocfg',
    'final_routes.rou.xml', 'low_routes.rou.xml',
    'medium_routes.rou.xml', 'high_routes.rou.xml',
]
print('\n── Required files check ──')
all_ok = True
for f in REQUIRED:
    exists = os.path.exists(f'/content/{f}')
    size   = f'{os.path.getsize(f"/content/{f}")/1024:.1f} KB' if exists else ''
    if not exists: all_ok = False
    print(f'  {"✓" if exists else "✗ MISSING":<12} {f:<35} {size}')

print()
print('✅ All set! Now run Cell 1.' if all_ok else '❌ Some files missing — check zip contents above!')

Cloning from GitHub...
✓ Repo cloned

Extracting Q-Learning.zip...
── Files inside zip ──
  final_routes.rou.xml
  gcn_encoder.py
  generate_density_routes.py
  high_routes.rou.xml
  low_routes.rou.xml
  ma_qlearning.py
  manhattan.net.xml
  manhattan_labels.poi.xml
  medium_routes.rou.xml
  run.sumocfg
  test_env.py
  traci_env_qlearning.py
✓ Extracted directly to /content

── Required files check ──
  ✓            gcn_encoder.py                      9.5 KB
  ✓            ma_qlearning.py                     12.2 KB
  ✓            traci_env_qlearning.py              22.1 KB
  ✓            manhattan.net.xml                   56.4 KB
  ✓            run.sumocfg                         0.3 KB
  ✓            final_routes.rou.xml                13.8 KB
  ✓            low_routes.rou.xml                  14.0 KB
  ✓            medium_routes.rou.xml               14.0 KB
  ✓            high_routes.rou.xml                 14.0 KB

✅ All set! Now run Cell 1.


# GCN + Multi-Agent Q-Learning
## Manhattan 4×4 Traffic Signal Control — Kaggle Training

**Run cells one by one in order. Do not skip.**

**Algorithm**: Linear Q-network · Online TD(0) · No replay buffer · No target network

| Cell | Purpose |
|------|---------|
| 1 | Install SUMO |
| 2 | Copy dataset files |
| 3 | Verify `traci_env_qlearning.py` is loaded |
| 4 | Install Python packages |
| 5 | Quick environment test (50 steps) |
| 6 | Stage 0 — Baseline (`final_routes`) |
| 7 | Stage 1 — Off-peak (`low_routes`) |
| 8 | Stage 2 — Moderate (`medium_routes`) |
| 9 | Stage 3 — Peak hour (`high_routes`) |
| 10 | Final evaluation across all densities |
| 11 | Plot training curves for all stages |

In [3]:
# ══════════════════════════════════════════════════════════════
# CELL 1 — Install SUMO (Internet must be ON, takes ~2-3 min)
# ══════════════════════════════════════════════════════════════
import subprocess, sys, os

print('Adding SUMO stable PPA...')
subprocess.run(['add-apt-repository', 'ppa:sumo/stable', '-y'],
               capture_output=True, text=True)

print('Updating apt...')
subprocess.run(['apt-get', 'update', '-q'],
               capture_output=True, text=True)

print('Installing SUMO...')
r = subprocess.run(
    ['apt-get', 'install', '-y', 'sumo', 'sumo-tools', 'sumo-doc'],
    capture_output=True, text=True
)
print(r.stdout[-500:] if r.stdout else '')
if r.returncode != 0:
    print('apt stderr:', r.stderr[-300:])

# Set SUMO_HOME BEFORE importing traci anywhere
os.environ['SUMO_HOME'] = '/usr/share/sumo'
if '/usr/share/sumo/tools' not in sys.path:
    sys.path.insert(0, '/usr/share/sumo/tools')

# Kill any stale SUMO processes
subprocess.run(['pkill', '-9', '-f', 'sumo'],  capture_output=True)
subprocess.run(['pkill', '-9', '-f', 'traci'], capture_output=True)

# Force clear any cached traci modules
for mod in list(sys.modules.keys()):
    if 'traci' in mod or 'sumo' in mod:
        del sys.modules[mod]

# Verify installation
ver = subprocess.run(['sumo', '--version'], capture_output=True, text=True)
if 'SUMO' in ver.stdout:
    print('\n✓ SUMO installed:', ver.stdout.split('\n')[0])
else:
    print('✗ ERROR: SUMO not found. Check stderr below:')
    print(ver.stderr)


Adding SUMO stable PPA...
Updating apt...
Installing SUMO...
k

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm_debug.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link



✓ SUMO installed: Eclipse SUMO sumo 1.26.0


In [4]:
# CELL 2: Files already in /content from setup cell — just verify
import os

WORK_DIR = '/content'
os.chdir(WORK_DIR)

print('Files already loaded from GitHub zip.')
print('\nFiles in /content:')
for f in sorted(os.listdir(WORK_DIR)):
    if not f.startswith('.') and os.path.isfile(os.path.join(WORK_DIR, f)):
        size = f'{os.path.getsize(os.path.join(WORK_DIR, f))/1024:.1f} KB'
        print(f'  {f:<40} {size}')

print('\n✓ Working directory ready — proceed to Cell 3')

Files already loaded from GitHub zip.

Files in /content:
  final_routes.rou.xml                     13.8 KB
  gcn_encoder.py                           9.5 KB
  generate_density_routes.py               7.5 KB
  high_routes.rou.xml                      14.0 KB
  low_routes.rou.xml                       14.0 KB
  ma_qlearning.py                          12.2 KB
  manhattan.net.xml                        56.4 KB
  manhattan_labels.poi.xml                 1.7 KB
  medium_routes.rou.xml                    14.0 KB
  run.sumocfg                              0.3 KB
  test_env.py                              5.8 KB
  traci_env_qlearning.py                   22.1 KB

✓ Working directory ready — proceed to Cell 3


In [7]:
# CELL 3: Verify traci_env_qlearning.py is loaded
import os
path = '/content/traci_env_qlearning.py'
if not os.path.exists(path):
    print('ERROR: traci_env_qlearning.py not found — upload it to your Kaggle dataset')
else:
    src = open(path).read()
    checks = {
        '_safe_close'    : 'safe SUMO close',
        'MAQLController' : 'Q-Learning controller import',
        'density'        : 'multi-density support',
        'resume_from'    : 'checkpoint resume',
    }
    all_ok = True
    for token, label in checks.items():
        found = token in src
        if not found:
            all_ok = False
        print(f'  {"OK" if found else "MISSING"}  {label} ({token!r})')
    print()
    if all_ok:
        print('traci_env_qlearning.py looks good — proceed to Cell 4')
    else:
        print('WARNING: re-upload the latest traci_env_qlearning.py')

  MISSING  safe SUMO close ('_safe_close')
  OK  Q-Learning controller import ('MAQLController')
  OK  multi-density support ('density')
  OK  checkpoint resume ('resume_from')



In [8]:
# CELL 4: Install Python packages
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'numpy', 'pandas', 'matplotlib'])
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {"CUDA" if torch.cuda.is_available() else "CPU"} ({gpu})')
print('Python packages ready')

PyTorch : 2.10.0+cpu
Device  : CPU (CPU only)
Python packages ready


In [9]:
# CELL 5: Quick environment test (50 steps only, not real training)
import os, sys, numpy as np
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from gcn_encoder import REAL_JUNCTIONS, RAW_OBS_DIM, ACT_DIM
from traci_env_qlearning import SUMOEnv, SUMO_CFG
#print(f'SUMO files dir : {_SCRIPTS_DIR}')
print(f'run.sumocfg    : {SUMO_CFG}')
print(f'Config exists  : {os.path.exists(SUMO_CFG)}')
print()
print('Running 50-step environment test...')
env = SUMOEnv(warmup=50, ep_duration=100)
obs = env.reset()
rewards = []
for _ in range(50):
    acts = {jid: np.random.randint(0, ACT_DIM[jid]) for jid in REAL_JUNCTIONS}
    obs, rews, done, info = env.step(acts)
    rewards.extend(rews.values())
    if done:
        break
env.close()
print(f'Reward range: min={min(rewards):.3f}  max={max(rewards):.3f}  mean={np.mean(rewards):.3f}')
print('Environment test PASSED' if max(rewards) <= 0 else 'WARNING: check reward range')

run.sumocfg    : run.sumocfg
Config exists  : True

Running 50-step environment test...
 Retrying in 1 seconds
Reward range: min=-0.194  max=-0.000  mean=-0.074
Environment test PASSED


In [10]:
# CELL 6: Stage 0 — Baseline training on final_routes.rou.xml
# Set EPISODES=5 to test first, then change to 300 for real training.
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from traci_env_qlearning import train
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 800      # change to 300 for full training
print(f'Algorithm : GCN + Multi-Agent Q-Learning')
print(f'Stage     : 0 — Baseline (final_routes)')
print(f'Device    : {DEVICE}  |  Episodes: {EPISODES}')
ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/content/ckpt_base',
    use_gui     = False,
    curriculum  = False,
    density     = None,     # uses final_routes.rou.xml (default)
)
print('\nStage 0 complete')

Algorithm : GCN + Multi-Agent Q-Learning
Stage     : 0 — Baseline (final_routes)
Device    : cpu  |  Episodes: 800
  GCN-MAQ-Learning Training  |  device=cpu  |  episodes=800
  Algorithm  : Q-Learning with Linear Q-network (no hidden layers)
  Reward     : -(0.5·NormQueue + 0.3·NormDelay + 0.2·NormWait)
  Update     : Online TD(0), no replay buffer, no target network
  Curriculum : OFF

  MAQLController ready
  Algorithm  : Q-Learning  |  Q-network: Linear (1 layer)
  Junctions  : 12
  GCN params : 6,560   Q-net per agent: 86   Total: 7,592
  Epsilon    : 1.0 → 0.05  (decay 0.995/episode)
  Update     : Online TD(0), no replay buffer, no target network
 Retrying in 1 seconds


KeyboardInterrupt: 

In [11]:
# Download Base Density Results
import os, shutil, zipfile
from google.colab import files as colab_files

ckpt_dir = '/content/ckpt_base'
log_path = f'{ckpt_dir}/training_log.csv'

# Create ZIP with Base density files
zip_path = '/content/base_density_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Training log
    if os.path.exists(log_path):
        zf.write(log_path, 'training_log.csv')
        print('Added: training_log.csv')

    # All checkpoint files
    if os.path.exists(ckpt_dir):
        for f in os.listdir(ckpt_dir):
            full_path = os.path.join(ckpt_dir, f)
            zf.write(full_path, f'checkpoint/{f}')
            print(f'Added: {f}')

print(f'\nZIP saved  : {zip_path}')
print(f'ZIP size   : {os.path.getsize(zip_path)/1024:.1f} KB')
display(FileLink('base_density_results.zip',
                 result_html_prefix='Download: '))


ZIP saved  : /content/base_density_results.zip
ZIP size   : 0.0 KB


NameError: name 'FileLink' is not defined

In [12]:
# CELL 7: Stage 1 — Off-peak training (low density, resumes from Stage 0)
# Set EPISODES=5 to test first, then change to 200 for real training.
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from traci_env_qlearning import train
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 1000      # change to 200 for full training
print(f'Algorithm : GCN + Multi-Agent Q-Learning')
print(f'Stage     : 1 — Off-peak (low density)')
print(f'Device    : {DEVICE}  |  Episodes: {EPISODES}')
ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/content/ckpt_low',
    resume_from = '/content/ckpt_base',
    use_gui     = False,
    curriculum  = False,
    density     = 'low',
)
print('\nStage 1 complete')

Algorithm : GCN + Multi-Agent Q-Learning
Stage     : 1 — Off-peak (low density)
Device    : cpu  |  Episodes: 1000
  GCN-MAQ-Learning Training  |  device=cpu  |  episodes=1000
  Algorithm  : Q-Learning with Linear Q-network (no hidden layers)
  Reward     : -(0.5·NormQueue + 0.3·NormDelay + 0.2·NormWait)
  Update     : Online TD(0), no replay buffer, no target network
  Curriculum : OFF

  MAQLController ready
  Algorithm  : Q-Learning  |  Q-network: Linear (1 layer)
  Junctions  : 12
  GCN params : 6,560   Q-net per agent: 86   Total: 7,592
  Epsilon    : 1.0 → 0.05  (decay 0.995/episode)
  Update     : Online TD(0), no replay buffer, no target network
  Loaded <- /content/ckpt_base/
  [SUMOEnv] density=low → low_routes.rou.xml
 Retrying in 1 seconds


KeyboardInterrupt: 

In [13]:
# Download Low Density Results
import os, shutil, zipfile
from google.colab import files as colab_files

ckpt_dir = '/content/ckpt_low'
log_path = f'{ckpt_dir}/training_log.csv'

# Create ZIP with low density files
zip_path = '/content/low_density_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Training log
    if os.path.exists(log_path):
        zf.write(log_path, 'training_log.csv')
        print('Added: training_log.csv')

    # All checkpoint files
    if os.path.exists(ckpt_dir):
        for f in os.listdir(ckpt_dir):
            full_path = os.path.join(ckpt_dir, f)
            zf.write(full_path, f'checkpoint/{f}')
            print(f'Added: {f}')

print(f'\nZIP saved  : {zip_path}')
print(f'ZIP size   : {os.path.getsize(zip_path)/1024:.1f} KB')
display(FileLink('low_density_results.zip',
                 result_html_prefix='Download: '))


ZIP saved  : /content/low_density_results.zip
ZIP size   : 0.0 KB


NameError: name 'FileLink' is not defined

In [14]:
# CELL 8: Stage 2 — Moderate traffic training (resumes from Stage 1)
# Set EPISODES=5 to test first, then change to 200 for real training.
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from traci_env_qlearning import train
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 1200      # change to 200 for full training
print(f'Algorithm : GCN + Multi-Agent Q-Learning')
print(f'Stage     : 2 — Moderate traffic (medium density)')
print(f'Device    : {DEVICE}  |  Episodes: {EPISODES}')
ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/content/ckpt_med',
    resume_from = '/content/weights/low',
    use_gui     = False,
    curriculum  = False,
    density     = 'medium',
)
print('\nStage 2 complete')

Algorithm : GCN + Multi-Agent Q-Learning
Stage     : 2 — Moderate traffic (medium density)
Device    : cpu  |  Episodes: 1200
  GCN-MAQ-Learning Training  |  device=cpu  |  episodes=1200
  Algorithm  : Q-Learning with Linear Q-network (no hidden layers)
  Reward     : -(0.5·NormQueue + 0.3·NormDelay + 0.2·NormWait)
  Update     : Online TD(0), no replay buffer, no target network
  Curriculum : OFF

  MAQLController ready
  Algorithm  : Q-Learning  |  Q-network: Linear (1 layer)
  Junctions  : 12
  GCN params : 6,560   Q-net per agent: 86   Total: 7,592
  Epsilon    : 1.0 → 0.05  (decay 0.995/episode)
  Update     : Online TD(0), no replay buffer, no target network
  Loaded <- /content/weights/low/
  [SUMOEnv] density=medium → medium_routes.rou.xml
 Retrying in 1 seconds


KeyboardInterrupt: 

In [15]:
# Download Medium Density Results
import os, shutil, zipfile
from google.colab import files as colab_files

ckpt_dir = '/content/ckpt_med'
log_path = f'{ckpt_dir}/training_log.csv'

# Create ZIP with Medium density files
zip_path = '/content/med_density_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Training log
    if os.path.exists(log_path):
        zf.write(log_path, 'training_log.csv')
        print('Added: training_log.csv')

    # All checkpoint files
    if os.path.exists(ckpt_dir):
        for f in os.listdir(ckpt_dir):
            full_path = os.path.join(ckpt_dir, f)
            zf.write(full_path, f'checkpoint/{f}')
            print(f'Added: {f}')

print(f'\nZIP saved  : {zip_path}')
print(f'ZIP size   : {os.path.getsize(zip_path)/1024:.1f} KB')
display(FileLink('med_density_results.zip',
                 result_html_prefix='Download: '))


ZIP saved  : /content/med_density_results.zip
ZIP size   : 0.0 KB


NameError: name 'FileLink' is not defined

In [16]:
# CELL 9: Stage 3 — Peak hour training (resumes from Stage 2)
# Set EPISODES=5 to test first, then change to 400 for real training.
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from traci_env_qlearning import train
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
EPISODES = 1500      # change to 400 for full training
print(f'Algorithm : GCN + Multi-Agent Q-Learning')
print(f'Stage     : 3 — Peak hour (high density)')
print(f'Device    : {DEVICE}  |  Episodes: {EPISODES}')
ctrl = train(
    n_episodes  = EPISODES,
    device      = DEVICE,
    save_dir    = '/content/ckpt_highf',
    resume_from = '/content/ckpt_high',
    use_gui     = False,
    curriculum  = False,
    density     = 'high',
)
print('\nStage 3 complete')

Algorithm : GCN + Multi-Agent Q-Learning
Stage     : 3 — Peak hour (high density)
Device    : cpu  |  Episodes: 1500
  GCN-MAQ-Learning Training  |  device=cpu  |  episodes=1500
  Algorithm  : Q-Learning with Linear Q-network (no hidden layers)
  Reward     : -(0.5·NormQueue + 0.3·NormDelay + 0.2·NormWait)
  Update     : Online TD(0), no replay buffer, no target network
  Curriculum : OFF

  MAQLController ready
  Algorithm  : Q-Learning  |  Q-network: Linear (1 layer)
  Junctions  : 12
  GCN params : 6,560   Q-net per agent: 86   Total: 7,592
  Epsilon    : 1.0 → 0.05  (decay 0.995/episode)
  Update     : Online TD(0), no replay buffer, no target network
  Loaded <- /content/ckpt_high/
  [SUMOEnv] density=high → high_routes.rou.xml
 Retrying in 1 seconds


KeyboardInterrupt: 

In [17]:
# Download High Density Results
import os, shutil, zipfile
from google.colab import files as colab_files

ckpt_dir = '/content/ckpt_high'
log_path = f'{ckpt_dir}/training_log.csv'

# Create ZIP with High density files
zip_path = '/content/high_density_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    # Training log
    if os.path.exists(log_path):
        zf.write(log_path, 'training_log.csv')
        print('Added: training_log.csv')

    # All checkpoint files
    if os.path.exists(ckpt_dir):
        for f in os.listdir(ckpt_dir):
            full_path = os.path.join(ckpt_dir, f)
            zf.write(full_path, f'checkpoint/{f}')
            print(f'Added: {f}')

print(f'\nZIP saved  : {zip_path}')
print(f'ZIP size   : {os.path.getsize(zip_path)/1024:.1f} KB')
display(FileLink('high_density_results.zip',
                 result_html_prefix='Download: '))


ZIP saved  : /content/high_density_results.zip
ZIP size   : 0.0 KB


NameError: name 'FileLink' is not defined

In [ ]:
# CELL 10: Final evaluation across all 3 densities with same model
# Loads best checkpoint (or final) and runs epsilon=0 evaluation.
import os, sys, torch
os.environ['SUMO_HOME'] = '/usr/share/sumo'
sys.path.insert(0, '/content')
sys.path.append('/usr/share/sumo/tools')
from traci_env_qlearning import _evaluate
from ma_qlearning import MAQLController
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Use best checkpoint if available, else final stage
CKPT_DIR = '/content/ckpt_high_best'
if not os.path.exists(CKPT_DIR):
    CKPT_DIR = '/content/ckpt_high'
print(f'Loading from: {CKPT_DIR}')
ctrl = MAQLController(device=DEVICE)
ctrl.load(CKPT_DIR)

print('\n=== Evaluation Results (same model on all densities) ===')
for density, label in [('low', 'Off-peak (low)'), ('medium', 'Moderate (medium)'), ('high', 'Peak hour (high)')]:
    print(f'\nDensity: {label}')
    _evaluate(ctrl, n_reps=3, use_gui=False, density=density)

In [ ]:
# CELL 11: Plot training curves for all stages
import pandas as pd, matplotlib.pyplot as plt, os

STAGES = [
    ('ckpt_base', 'Stage 0: Baseline',    '#7B9ED9'),
    ('ckpt_low',  'Stage 1: Off-peak',    '#5BA4E5'),
    ('ckpt_med',  'Stage 2: Moderate',    '#F0A050'),
    ('ckpt_high', 'Stage 3: Peak hour',   '#E24B4A'),
]

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('GCN + Multi-Agent Q-Learning — Training Curves (All Stages)',
             fontsize=14, fontweight='bold', y=1.01)

for ax, (ckpt, title, color) in zip(axes.flat, STAGES):
    log_path = f'/content/{ckpt}/training_log.csv'
    if os.path.exists(log_path):
        df = pd.read_csv(log_path)
        w  = max(1, len(df) // 20)

        # Reward (left axis)
        smooth_r = df['avg_reward'].rolling(w, min_periods=1).mean()
        ax.fill_between(df['episode'], df['avg_reward'], alpha=0.10, color=color)
        ax.plot(df['episode'], df['avg_reward'], color=color, alpha=0.25, lw=0.8)
        ax.plot(df['episode'], smooth_r,         color=color, lw=2.2, label='Reward')
        ax.set_ylabel('Avg Reward', color=color)
        ax.tick_params(axis='y', labelcolor=color)

        # Travel time (right axis)
        ax2 = ax.twinx()
        smooth_tt = df['avg_travel_time'].rolling(w, min_periods=1).mean()
        ax2.fill_between(df['episode'], df['avg_travel_time'], alpha=0.06, color='#888888')
        ax2.plot(df['episode'], df['avg_travel_time'], color='#AAAAAA', alpha=0.25, lw=0.8)
        ax2.plot(df['episode'], smooth_tt,             color='#555555', lw=1.6,
                 linestyle='--', label='Travel time')
        ax2.set_ylabel('Travel Time (s)', color='#555555')
        ax2.tick_params(axis='y', labelcolor='#555555')

        # Legend
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='lower right')

        ax.set_title(title, fontsize=11, pad=6)
        ax.set_xlabel('Episode')
        ax.grid(True, alpha=0.2, linestyle='--')
        ax.spines[['top']].set_visible(False)
    else:
        ax.text(0.5, 0.5, f'No data\n{ckpt}', ha='center', va='center',
                transform=ax.transAxes, fontsize=10, color='#888888')
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Episode')

plt.tight_layout()
save_path = '/content/qlearning_training_curves_all_stages.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved: {save_path}')

In [ ]:
# CELL 12: Collect ALL points + BEST points and download as ZIP
import os, pandas as pd, json, zipfile, shutil
from google.colab import files as colab_files

STAGES = [
    ('ckpt_base', 'baseline', None),
    ('ckpt_low',  'low',      'low'),
    ('ckpt_med',  'medium',   'medium'),
    ('ckpt_highf', 'high',     'high'),
]

all_records  = []   # every episode from every stage
best_records = []   # only the best episode per stage
summary      = []   # one-line summary per stage

for ckpt, stage_name, density in STAGES:
    log_path = f'/content/{ckpt}/training_log.csv'
    if not os.path.exists(log_path):
        print(f'[SKIP] {ckpt} — no training_log.csv found')
        continue

    df = pd.read_csv(log_path)
    df['stage']   = stage_name
    df['density'] = density if density else 'baseline'

    # --- ALL points ---
    all_records.append(df)

    # --- BEST point (highest avg_reward episode) ---
    best_row = df.loc[df['avg_reward'].idxmax()].copy()
    best_records.append(best_row)

    # --- Summary stats ---
    summary.append({
        'stage'            : stage_name,
        'density'          : density if density else 'baseline',
        'total_episodes'   : len(df),
        'best_reward'      : round(df['avg_reward'].max(), 4),
        'best_episode'     : int(df.loc[df['avg_reward'].idxmax(), 'episode']),
        'final_reward'     : round(df['avg_reward'].iloc[-1], 4),
        'mean_reward'      : round(df['avg_reward'].mean(), 4),
        'best_travel_time' : round(df['avg_travel_time'].min(), 2) if 'avg_travel_time' in df.columns else None,
        'mean_travel_time' : round(df['avg_travel_time'].mean(), 2) if 'avg_travel_time' in df.columns else None,
    })
    print(f'[OK] {stage_name:8s} | episodes={len(df)} | best_reward={best_row["avg_reward"]:.4f}')

# Save CSVs
out = '/content/qlearning_results'
os.makedirs(out, exist_ok=True)

all_df  = pd.concat(all_records, ignore_index=True)
best_df = pd.DataFrame(best_records).reset_index(drop=True)
sum_df  = pd.DataFrame(summary)

all_df.to_csv(f'{out}/all_points.csv',   index=False)
best_df.to_csv(f'{out}/best_points.csv', index=False)
sum_df.to_csv(f'{out}/summary.csv',      index=False)

print(f'\nAll points  : {len(all_df)} rows  → all_points.csv')
print(f'Best points : {len(best_df)} rows  → best_points.csv')
print(f'Summary     : {len(sum_df)} rows  → summary.csv')

# Print summary table
print('\n=== SUMMARY ===')
print(sum_df.to_string(index=False))

# ZIP everything
zip_path = '/content/qlearning_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['all_points.csv', 'best_points.csv', 'summary.csv']:
        zf.write(f'{out}/{fname}', fname)

print(f'\nZIP saved: {zip_path}')
print(f'ZIP size : {os.path.getsize(zip_path)/1024:.1f} KB')
display(FileLink('/content/qlearning_results.zip', result_html_prefix='Download: '))

In [ ]:
import shutil, os

# Only copy individual CSVs to working directory
for f in ['all_points.csv', 'best_points.csv', 'summary.csv']:
    src = f'/content/qlearning_results/{f}'
    dst = f'/content/{f}'
    shutil.copy(src, dst)
    print(f'Copied: {f}')

print('\nDone! Now go to Output panel (right side) and download qlearning_results.zip')